# Part 1 — Building the Frozen Inputs

This notebook walks through the Part-1 pipeline end to end:

| Step | Section | Output |
|------|---------|--------|
| Connectome | 4b–4e | `W_dk68.npy`, `region_names.npy` |
| FTD atrophy | 5b | `atrophy_FTD.npy` *(pending)* |
| AD atrophy | 5c | `atrophy_AD.npy` *(pending)* |

All source code lives in `src/levers/data.py`. Once the four `.npy` files are written to `data/processed/` they are treated as **frozen inputs** — later analysis notebooks read them directly and never re-run this pipeline.

---
**Data source:** Budapest connectome v3 (HCP, 1064 subjects, Lausanne scale33 = 83 nodes)  
**Atlas:** Desikan–Killiany 68-region cortical parcellation (DK-68)  
**Weight:** mean number of streamlines across subjects, spectral-radius normalized

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import networkx as nx

# make src/ importable without an editable install
ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

from levers.data import DK34, CANON, build_connectome

PROCESSED = ROOT / 'data' / 'processed'
RAW       = ROOT / 'data' / 'raw'

## 1  Load the processed connectome

If `data/processed/W_dk68.npy` already exists (built by `scripts/01_build_inputs.py`), we load it directly. Otherwise we run the pipeline here — this will take ~2 minutes while it reads 1064 GraphML files.

In [ ]:
W_path = PROCESSED / 'W_dk68.npy'
N_path = PROCESSED / 'region_names.npy'

if W_path.exists() and N_path.exists():
    W     = np.load(W_path)
    names = list(np.load(N_path, allow_pickle=True))
    print(f'Loaded from cache: {W_path}')
else:
    print('Cache missing — running pipeline (grab a coffee)...')
    graphml_dir = RAW / 'budapest_83.graphml'
    W, names = build_connectome(graphml_path=graphml_dir, out_dir=PROCESSED)

print(f'W shape      : {W.shape}')
print(f'Spectral rad : {np.max(np.linalg.eigvals(W).real):.6f}  (normalized to 1.0)')
print(f'Symmetric    : {np.allclose(W, W.T)}')
print(f'Non-zero edges: {(W > 0).sum() // 2}')

## 2  Connectome heatmap

The 68 regions are ordered left hemisphere first (rows/cols 0–33), then right (34–67). The block structure reflects inter- vs intra-hemispheric connectivity.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

im = ax.imshow(W, cmap='hot', aspect='auto', interpolation='nearest')
plt.colorbar(im, ax=ax, label='Normalized fiber count')

# hemisphere divider
ax.axhline(33.5, color='cyan', lw=1, ls='--', alpha=0.7)
ax.axvline(33.5, color='cyan', lw=1, ls='--', alpha=0.7)
ax.text(17, -2.5, 'Left hemisphere', ha='center', va='bottom', fontsize=9, color='cyan')
ax.text(51, -2.5, 'Right hemisphere', ha='center', va='bottom', fontsize=9, color='cyan')

# tick labels: bare region name every 4 rows to avoid crowding
tick_pos   = list(range(0, 68, 4))
tick_label = [names[i].split('_', 1)[1] for i in tick_pos]
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_label, rotation=90, fontsize=7)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_label, fontsize=7)

ax.set_title('DK-68 group connectome\n(Budapest v3, 1064 HCP subjects, spectral-radius normalized)', fontsize=11)
fig.tight_layout()
plt.savefig(PROCESSED / 'W_dk68_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to data/processed/W_dk68_heatmap.png')

## 3  Degree distribution

Weighted node degree (sum of fiber counts per region) highlights which regions are structural hubs.

In [ ]:
degree = W.sum(axis=1)
order  = np.argsort(degree)[::-1]

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#4878CF'] * 34 + ['#D65F5F'] * 34   # blue=LH, red=RH
ax.bar(range(68), degree[order], color=[colors[i] for i in order], width=0.8)
ax.set_xticks(range(68))
ax.set_xticklabels([names[i] for i in order], rotation=90, fontsize=6)
ax.set_ylabel('Weighted degree (sum of normalized fibers)')
ax.set_title('Regional weighted degree — structural hubs (blue = LH, red = RH)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#4878CF', label='Left'), Patch(color='#D65F5F', label='Right')], fontsize=9)
fig.tight_layout()
plt.savefig(PROCESSED / 'degree_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4  Region ordering (canonical DK-68 index)

Every vector in this project uses this exact ordering. Index 0–33 = left hemisphere, 34–67 = right.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'index': range(68),
    'name': names,
    'hemisphere': ['Left'] * 34 + ['Right'] * 34,
    'bare_region': [n.split('_', 1)[1] for n in names],
    'weighted_degree': degree,
})
df.set_index('index', inplace=True)
df.sort_values('weighted_degree', ascending=False).head(20)

## 5  Basic graph statistics

In [ ]:
# threshold at > 0 for binary graph stats
G = nx.from_numpy_array(W)

print(f"Nodes                : {G.number_of_nodes()}")
print(f"Edges (W > 0)        : {G.number_of_edges()}")
print(f"Density              : {nx.density(G):.3f}")
print(f"Mean clustering coef : {nx.average_clustering(G, weight='weight'):.4f}")
print(f"Max edge weight      : {W.max():.4f}")
print(f"Mean edge weight (W>0): {W[W>0].mean():.4f}")

eigvals = np.linalg.eigvals(W).real
print(f"Spectral radius      : {eigvals.max():.6f}")
print(f"Second eigenvalue    : {np.sort(eigvals)[-2]:.6f}")

---
## Next steps

- **Section 5b** — parcellate `ftd_bvFTD_tmap.nii.gz` (Zenodo 10383493) onto DK-68 → `atrophy_FTD.npy`
- **Section 5c** — derive AD atrophy vector from ADNI cortical thickness tables → `atrophy_AD.npy`
- **Part 2+** — network control theory analysis using `W_dk68`, `atrophy_AD`, `atrophy_FTD`